# **WEEK-4 Data Cleaning and Preparation**

goal :
- Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate)
- Remove unnecessary or redundant columns
- Handle missing values appropriately
- Ensure numeric fields are properly typed
- Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms

### Set Up

In [1]:
from pathlib import Path
from datetime import datetime
import re
import pandas as pd
import os

DATA_DIR = Path("/Users/kmaxx/Desktop/IDX-da/idx_data")

sold_df = pd.read_csv(DATA_DIR / "sold_with_rates.csv", low_memory= False)
listing_df = pd.read_csv(DATA_DIR / "listing_with_rates.csv", low_memory= False)

### Data Convertion and Cleaning

In [10]:
def convert_date_fields(df):
    date_cols = [
        "CloseDate",
        "PurchaseContractDate",
        "ListingContractDate",
        "ContractStatusChangeDate"
    ]

    df = df.copy()
    df.columns = df.columns.str.strip()

    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df

In [5]:
sold_df = convert_date_fields(sold_df)
listing_df = convert_date_fields(listing_df)

In [12]:
def remove_invalid_numeric_values(df):
    data = df.copy()

    numeric_cols = [
        "ClosePrice",
        "LivingArea",
        "DaysOnMarket",
        "BedroomsTotal",
        "BathroomsTotalInteger"
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    valid_mask = (
        (data["ClosePrice"] > 0) &
        (data["LivingArea"] > 0) &
        (data["DaysOnMarket"] >= 0) &
        (data["BedroomsTotal"] >= 0) &
        (data["BathroomsTotalInteger"] >= 0)
    )

    cleaned_df = data[valid_mask].copy()

    print(f"Rows before cleaning: {len(data):,}")
    print(f"Rows after cleaning:  {len(cleaned_df):,}")
    print(f"Rows removed:         {len(data) - len(cleaned_df):,}")

    return cleaned_df

In [13]:
sold_df = remove_invalid_numeric_values(sold_df)

Rows before cleaning: 384,216
Rows after cleaning:  383,761
Rows removed:         455


In [14]:
listing_df = remove_invalid_numeric_values(listing_df)

Rows before cleaning: 567,549
Rows after cleaning:  140,454
Rows removed:         427,095


### Date Consistency Checks

In [17]:
def add_date_consistency_flags(df):
    data = df.copy()

    date_cols = [
        "ListingContractDate",
        "PurchaseContractDate",
        "CloseDate",
        "ContractStatusChangeDate"
    ]

    for col in date_cols:
        if col in data.columns:
            data[col] = pd.to_datetime(data[col], errors="coerce")

    data["listing_after_close_flag"] = (
        data["ListingContractDate"] > data["CloseDate"]
    )

    data["purchase_after_close_flag"] = (
        data["PurchaseContractDate"] > data["CloseDate"]
    )

    data["negative_timeline_flag"] = (
        (data["ListingContractDate"] > data["PurchaseContractDate"]) |
        (data["PurchaseContractDate"] > data["CloseDate"]) |
        (data["ListingContractDate"] > data["CloseDate"])
    )

    return data

In [18]:
sold_df = add_date_consistency_flags(sold_df)
listing_df = add_date_consistency_flags(listing_df)

### Geographic Data Checks

In [19]:
def add_geographic_flags(df):
    data = df.copy()

    data["Latitude"] = pd.to_numeric(data["Latitude"], errors="coerce")
    data["Longitude"] = pd.to_numeric(data["Longitude"], errors="coerce")

    # Missing coordinates
    data["missing_coordinates_flag"] = (
        data["Latitude"].isna() | data["Longitude"].isna()
    )

    # Sentinel null values
    data["zero_coordinates_flag"] = (
        (data["Latitude"] == 0) | (data["Longitude"] == 0)
    )

    # California longitudes should be negative
    data["positive_longitude_flag"] = (
        data["Longitude"] > 0
    )

    # Rough California coordinate bounds
    data["implausible_coordinates_flag"] = (
        (data["Latitude"] < 32) |
        (data["Latitude"] > 42) |
        (data["Longitude"] < -125) |
        (data["Longitude"] > -114)
    )

    return data

In [20]:
sold_df = add_date_consistency_flags(sold_df)
listing_df = add_date_consistency_flags(listing_df)

### Export

In [ ]:
sold_df.to_csv(DATA_DIR / "cleaned_sold", )